# Phase 2B - Golden Tables Ctreation

![image_1783071767775.png](./image_1783071767775.png "image_1783071767775.png")

In [0]:
from pyspark.sql.functions import col, sum as _sum, round as _round, count, avg
from pyspark.sql.window import Window

items = spark.table("workspace.silver.order_items_enriched")
orders_clean = spark.table("workspace.silver.orders_clean")

### Gold T1 - daily_revenue_by_category: daily revenue by category
- Excludes canceled/unavailable orders — we don't book revenue we never earned.
- Sums `price` per day per category → the trend view for the business.
- **OPTIMIZE** = compacts many small Delta files into fewer large ones (faster reads).
- **ZORDER BY (date)** = physically groups rows with nearby dates together, so date-filtered queries scan fewer files. No visible speed change at this size — it's the technique that matters at scale.

In [0]:
daily_revenue_by_category = (items
    .filter(~col("order_status").isin("canceled", "unavailable"))      # don't count revenue never earned
    .groupBy("order_purchase_date", "product_category")
    .agg(_round(_sum("price"), 2).alias("revenue"))
    .orderBy("order_purchase_date", "product_category"))

daily_revenue_by_category.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.daily_revenue_by_category")

spark.sql("OPTIMIZE workspace.gold.daily_revenue_by_category ZORDER BY (order_purchase_date)")

print("gold.daily_revenue_by_category rows:", spark.table("workspace.gold.daily_revenue_by_category").count())
spark.table("workspace.gold.daily_revenue_by_category").show(5)

gold.daily_revenue_by_category rows: 18943
+-------------------+----------------+-------+
|order_purchase_date|product_category|revenue|
+-------------------+----------------+-------+
|         2016-09-04|moveis_decoracao|  72.89|
|         2016-09-15|    beleza_saude| 134.97|
|         2016-10-03|      brinquedos|  128.9|
|         2016-10-03|   esporte_lazer|  58.39|
|         2016-10-03|fashion_calcados|  29.99|
+-------------------+----------------+-------+
only showing top 5 rows


In [0]:
top_customers

---------------------------------------------------------------------------
NameError Traceback (most recent call last)
File , line 1
----> 1 top_customers

NameError: name 'top_customers' is not defined

### Gold T2 - top_customers: top 10 customers by lifetime value
- Grouped by **`customer_unique_id`**, NOT `customer_id` — otherwise every customer looks like a one-order customer and the ranking is meaningless.
- Sums each person's spend, ranks, keeps the top 10 → the retention/marketing shortlist.

In [0]:
top_customers = (items
    .filter(~col("order_status").isin("canceled", "unavailable"))
    .groupBy("customer_unique_id")                                     # the real person, NOT customer_id
    .agg(_round(_sum("price"), 2).alias("lifetime_value"),
         count("order_id").alias("items_bought"))
    .orderBy(col("lifetime_value").desc())
    .limit(10))

top_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.top_customers")

print("gold.top_customers rows:", spark.table("workspace.gold.top_customers").count())
spark.table("workspace.gold.top_customers").show(truncate=False)

gold.top_customers rows: 10
+--------------------------------+--------------+------------+
|customer_unique_id              |lifetime_value|items_bought|
+--------------------------------+--------------+------------+
|0a0a92112bd4c708ca5fde585afaa872|13440.0       |8           |
|da122df9eeddfedc1dc1f5349a1a690c|7388.0        |2           |
|763c8b1c9c68a0229c42c9fc6f662b93|7160.0        |4           |
|dc4802a71eae9be1dd28f5d788ceb526|6735.0        |1           |
|459bef486812aa25204be022145caa62|6729.0        |1           |
|ff4159b92c40ebe40454e3e6a7c35ed6|6499.0        |1           |
|4007669dec559734d6f53e029e360987|5934.6        |6           |
|eebb5dda148d3893cdaf5b5ca3040ccb|4690.0        |1           |
|48e1ac109decbb87765a3eade6854098|4590.0        |1           |
|a229eba70ec1c2abef51f04987deb7a5|4400.0        |2           |
+--------------------------------+--------------+------------+



### Gold T3 - daily_orders: 7-day rolling average order volume
- Counts orders per day, then averages each day with the previous 6 → a smoothed demand line that hides daily spikes.
- **Window function** = a calculation over a sliding set of rows; `rowsBetween(-6, 0)` = today plus the last 6 days.
- Order grain (counting orders, not items), then Z-ordered by date.

In [0]:
daily_orders = (orders_clean
    .groupBy("order_purchase_date")
    .agg(count("order_id").alias("order_count")))

w = Window.orderBy("order_purchase_date").rowsBetween(-6, 0)           # today + previous 6 days

rolling_7day = (daily_orders
    .withColumn("rolling_7day_avg_orders", _round(avg("order_count").over(w), 1))
    .orderBy("order_purchase_date"))

rolling_7day.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.rolling_7day_order_volume")

spark.sql("OPTIMIZE workspace.gold.rolling_7day_order_volume ZORDER BY (order_purchase_date)")

print("gold.rolling_7day_order_volume rows:", spark.table("workspace.gold.rolling_7day_order_volume").count())
spark.table("workspace.gold.rolling_7day_order_volume").show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.rolling_7day_order_volume rows: 634
+-------------------+-----------+-----------------------+
|order_purchase_date|order_count|rolling_7day_avg_orders|
+-------------------+-----------+-----------------------+
|         2016-09-04|          1|                    1.0|
|         2016-09-05|          1|                    1.0|
|         2016-09-13|          1|                    1.0|
|         2016-09-15|          1|                    1.0|
|         2016-10-02|          1|                    1.0|
|         2016-10-03|          8|                    2.2|
|         2016-10-04|         63|                   10.9|
|         2016-10-05|         47|                   17.4|
|         2016-10-06|         51|                   24.6|
|         2016-10-07|         46|                   31.0|
+-------------------+-----------+-----------------------+
only showing top 10 rows
